Ex. 23

In [1]:
class DSU:
    def __init__(self, n):
        self.p, self.r = list(range(n)), [0] * n
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, x, y):
        x, y = self.find(x), self.find(y)
        if x == y: return False
        if self.r[x] < self.r[y]: x, y = y, x
        self.p[y] = x
        if self.r[x] == self.r[y]: self.r[x] += 1
        return True

def kruskal(n, edges):
    d = DSU(n)
    return [e for e in sorted(edges, key=lambda e: e[2]) if d.union(e[0], e[1])]

def bot(t): return max(w for _, _, w in t)
def cost(t): return sum(w for _, _, w in t)
def is_span(n, es):
    if len(es) != n - 1: return False
    d = DSU(n)
    return all(d.union(u, v) for u, v, w in es)

from itertools import combinations

def min_bot(n, edges):
    return min(bot(list(s)) for s in combinations(edges, n-1) if is_span(n, list(s)))

n = 3
E = [(0, 1, 1), (1, 2, 2), (0, 2, 2)]
T = kruskal(n, E)
alt = [(0, 2, 2), (1, 2, 2)]

print(f"MST : {T}  cost={cost(T)}  bot={bot(T)}")
print(f"Alt : {alt}  cost={cost(alt)}  bot={bot(alt)}")
print(f"(a) alt — тоже MBST (bot={bot(alt)}), но не MST (cost {cost(alt)} > {cost(T)}): контрпример")
print(f"(b) bot(MST)={bot(T)} == min_bot={min_bot(n, E)}: MST всегда является MBST")

MST : [(0, 1, 1), (1, 2, 2)]  cost=3  bot=2
Alt : [(0, 2, 2), (1, 2, 2)]  cost=4  bot=2
(a) alt — тоже MBST (bot=2), но не MST (cost 4 > 3): контрпример
(b) bot(MST)=2 == min_bot=2: MST всегда является MBST


Ex. 24

In [3]:
from collections import defaultdict

class DSU:
    def __init__(self, n):
        self.p, self.r = list(range(n)), [0] * n
    def find(self, x):
        if self.p[x] != x: self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, x, y):
        x, y = self.find(x), self.find(y)
        if x == y: return False
        if self.r[x] < self.r[y]: x, y = y, x
        self.p[y] = x
        if self.r[x] == self.r[y]: self.r[x] += 1
        return True

def kruskal(n, edges):
    d = DSU(n)
    return [e for e in sorted(edges, key=lambda e: e[2]) if d.union(e[0], e[1])]

def cost(t): return sum(w for _, _, w in t)

def tadj(tree):
    g = defaultdict(list)
    for u, v, w in tree:
        g[u].append((v, w)); g[v].append((u, w))
    return g

def path_max(tree, s, t):
    g = tadj(tree)
    vis, prev, ew = {s}, {s: None}, {}
    stk = [s]
    while stk:
        u = stk.pop()
        for v, w in g[u]:
            if v not in vis:
                vis.add(v); prev[v] = u
                ew[(min(u,v), max(u,v))] = w
                stk.append(v)
    mx, me = 0, None
    cur = t
    while prev[cur] is not None:
        e = (min(cur, prev[cur]), max(cur, prev[cur]))
        if ew[e] > mx: mx, me = ew[e], e
        cur = prev[cur]
    return mx, me

def upd(n, tree, u, v, c):
    mx, me = path_max(tree, u, v)
    if c >= mx:
        return tree, False
    nt = [e for e in tree if (min(e[0],e[1]), max(e[0],e[1])) != me]
    return nt + [(u, v, c)], True

n = 5
E = [(0,1,4),(0,2,3),(1,2,1),(1,3,2),(2,4,5),(3,4,7)]
T = kruskal(n, E)

print(f"Исходный MST : {sorted(T)}  cost={cost(T)}")
T2, ch = upd(n, T, 0, 4, 2)
print(f"+ ребро(0,4,2) : изменён={ch}  новый MST={sorted(T2)}  cost={cost(T2)}")
T3, ch = upd(n, T, 0, 4, 10)
print(f"+ ребро(0,4,10): изменён={ch}  MST не изменился")

Исходный MST : [(0, 2, 3), (1, 2, 1), (1, 3, 2), (2, 4, 5)]  cost=11
+ ребро(0,4,2) : изменён=True  новый MST=[(0, 2, 3), (0, 4, 2), (1, 2, 1), (1, 3, 2)]  cost=8
+ ребро(0,4,10): изменён=False  MST не изменился


Ex. 25

In [5]:
from itertools import groupby, permutations

class DSU:
    def __init__(self, n):
        self.p, self.r = list(range(n)), [0] * n
    def find(self, x):
        if self.p[x] != x: self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, x, y):
        x, y = self.find(x), self.find(y)
        if x == y: return False
        if self.r[x] < self.r[y]: x, y = y, x
        self.p[y] = x
        if self.r[x] == self.r[y]: self.r[x] += 1
        return True

def kruskal(n, edges):
    d = DSU(n)
    return [e for e in sorted(edges, key=lambda e: e[2]) if d.union(e[0], e[1])]

def cost(t): return sum(w for _, _, w in t)

def run(n, order):
    d = DSU(n)
    t = [e for e in order if d.union(e[0], e[1])]
    return tuple(sorted((min(u,v), max(u,v), w) for u, v, w in t))

def all_mst(n, edges):
    grps = [list(g) for _, g in groupby(sorted(edges, key=lambda e: e[2]), key=lambda e: e[2])]
    orders = [[]]
    for g in grps:
        orders = [o + list(p) for o in orders for p in permutations(g)]
    seen, res = set(), []
    for o in orders:
        t = run(n, o)
        if len(t) == n-1 and t not in seen:
            seen.add(t); res.append(list(t))
    return res

def order_for(n, edges, T):
    ts = {(min(u,v), max(u,v)) for u, v, w in T}
    return sorted(edges, key=lambda e: (e[2], 0 if (min(e[0],e[1]), max(e[0],e[1])) in ts else 1))

n = 4
E = [(0,1,1),(1,2,1),(0,2,1),(2,3,2),(1,3,2)]
T = kruskal(n, E)
T_canon = run(n, T)
reproduced = run(n, order_for(n, E, T))

print(f"MST T: {sorted(T)}")
print(f"Kruskal с построенным порядком воспроизводит T: {reproduced == T_canon}")

trees = all_mst(n, E)
print(f"\nВсе MST ({len(trees)} шт.), cost={cost(trees[0])} у каждого:")
for t in trees:
    print(f"  {sorted(t)}")

MST T: [(0, 1, 1), (1, 2, 1), (2, 3, 2)]
Kruskal с построенным порядком воспроизводит T: True

Все MST (6 шт.), cost=4 у каждого:
  [(0, 1, 1), (1, 2, 1), (2, 3, 2)]
  [(0, 1, 1), (1, 2, 1), (1, 3, 2)]
  [(0, 1, 1), (0, 2, 1), (2, 3, 2)]
  [(0, 1, 1), (0, 2, 1), (1, 3, 2)]
  [(0, 2, 1), (1, 2, 1), (2, 3, 2)]
  [(0, 2, 1), (1, 2, 1), (1, 3, 2)]
